### Import the necessary database

In [1]:
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

In [2]:
#In[2]:
# define function
import src.SAT_function_Obs_Fingerprint as data_process
import src.Data_Preprocess as preprosess

In [3]:
# import src.slurm_cluster as scluster
# client, scluster = scluster.init_dask_slurm_cluster(scale=4, cores=50, memory="200GB")

In [4]:
# read all segmented trend patterns (L=10..73) into a dict of DataArrays
import os
dir_ICV_seg_TREND = '/work/mh0033/m301036/OBS_LPS_revision/docs/data/FIG3/OBS_ICV_std/'
len_segments = np.arange(10, 74, 1)

ds = {}
for L in len_segments:
    fpath = os.path.join(dir_ICV_seg_TREND, f"OBS_ICV_MK_trend_segments_L{L}.nc")
    da = xr.open_dataset(fpath, chunks={"lat": 10, "lon": 10})['trend']
    ds[f"ICV_trend_{L}yr"] = da


In [5]:
#Perfrom land sea mask
land_sea_mask=xr.open_dataset('/work/mh0033/m301036/Data_storage/CMIP6-MPI-ESM-LR/GR15_lsm_regrid.nc')
# land_sea_mask.coords
land_sea_mask

# mask the land area trend data
mask_data = land_sea_mask['var1']
mask_data

<xarray.DataArray 'var1' (time: 1, lat: 90, lon: 180)>
[16200 values with dtype=float32]
Coordinates:
  * time     (time) float64 201.0
  * lon      (lon) float64 0.0 2.0 4.0 6.0 8.0 ... 350.0 352.0 354.0 356.0 358.0
  * lat      (lat) float64 -89.0 -87.0 -85.0 -83.0 -81.0 ... 83.0 85.0 87.0 89.0
Attributes:
    code:     1

In [6]:
ds_masked = {}
for key in ds.keys():
    ds_masked[key] = ds[key].where(mask_data[0,:,:]==0, drop = False)

In [7]:
ds_masked_adj = {}
for key in ds_masked.keys():
    ds_masked_adj[key] = preprosess.convert_longitude(ds_masked[key])

In [9]:
ds_adj = {}
for key in ds.keys():
    ds_adj[key] = preprosess.convert_longitude(ds[key])

In [10]:
ds_adj

{'ICV_trend_10yr': <xarray.DataArray 'trend' (segment: 164, lat: 90, lon: 180)>
 dask.array<getitem, shape=(164, 90, 180), dtype=float64, chunksize=(164, 10, 10), chunktype=numpy.ndarray>
 Coordinates:
   * lat      (lat) float64 -89.0 -87.0 -85.0 -83.0 -81.0 ... 83.0 85.0 87.0 89.0
   * lon      (lon) float64 -180.0 -178.0 -176.0 -174.0 ... 174.0 176.0 178.0
 Dimensions without coordinates: segment,
 'ICV_trend_11yr': <xarray.DataArray 'trend' (segment: 163, lat: 90, lon: 180)>
 dask.array<getitem, shape=(163, 90, 180), dtype=float64, chunksize=(163, 10, 10), chunktype=numpy.ndarray>
 Coordinates:
   * lat      (lat) float64 -89.0 -87.0 -85.0 -83.0 -81.0 ... 83.0 85.0 87.0 89.0
   * lon      (lon) float64 -180.0 -178.0 -176.0 -174.0 ... 174.0 176.0 178.0
 Dimensions without coordinates: segment,
 'ICV_trend_12yr': <xarray.DataArray 'trend' (segment: 162, lat: 90, lon: 180)>
 dask.array<getitem, shape=(162, 90, 180), dtype=float64, chunksize=(162, 10, 10), chunktype=numpy.ndarray>
 Coo

In [8]:
ds_masked_adj

{'ICV_trend_10yr': <xarray.DataArray 'trend' (segment: 164, lat: 90, lon: 180)>
 dask.array<getitem, shape=(164, 90, 180), dtype=float64, chunksize=(164, 10, 10), chunktype=numpy.ndarray>
 Coordinates:
   * lat      (lat) float64 -89.0 -87.0 -85.0 -83.0 -81.0 ... 83.0 85.0 87.0 89.0
   * lon      (lon) float64 -180.0 -178.0 -176.0 -174.0 ... 174.0 176.0 178.0
     time     float64 201.0
 Dimensions without coordinates: segment,
 'ICV_trend_11yr': <xarray.DataArray 'trend' (segment: 163, lat: 90, lon: 180)>
 dask.array<getitem, shape=(163, 90, 180), dtype=float64, chunksize=(163, 10, 10), chunktype=numpy.ndarray>
 Coordinates:
   * lat      (lat) float64 -89.0 -87.0 -85.0 -83.0 -81.0 ... 83.0 85.0 87.0 89.0
   * lon      (lon) float64 -180.0 -178.0 -176.0 -174.0 ... 174.0 176.0 178.0
     time     float64 201.0
 Dimensions without coordinates: segment,
 'ICV_trend_12yr': <xarray.DataArray 'trend' (segment: 162, lat: 90, lon: 180)>
 dask.array<getitem, shape=(162, 90, 180), dtype=float64

In [11]:
lat = ds_masked_adj["ICV_trend_10yr"].lat
lon = ds_masked_adj["ICV_trend_10yr"].lon
# Extratropical South Pacific region
lat1 = 42
lat2 = 60
lon1 = -50
lon2 = -10
def calc_North_Atlantic_anomalies(data,mask_data):
    ds_WH = data.sel(lat=slice(42, 60), lon=slice(-50, -10))
    ds_WH_anomaly = data_process.calc_weighted_mean(ds_WH)
    ds_sel = mask_data.sel(lat=slice(0, 90), lon=slice(-180,180))
    ds_sel_anomaly = data_process.calc_weighted_mean(ds_sel)
    
    ds_anomalies = ds_WH_anomaly - ds_sel_anomaly
    return ds_anomalies

In [12]:
ds_subpolar_gyre_anom = {}
for key in ds_masked_adj.keys():
    ds_subpolar_gyre_anom[key] = calc_North_Atlantic_anomalies(ds_adj[key], ds_masked_adj[key])

In [13]:
ds_subpolar_gyre_anom

{'ICV_trend_10yr': <xarray.DataArray 'trend' (segment: 164)>
 dask.array<sub, shape=(164,), dtype=float64, chunksize=(164,), chunktype=numpy.ndarray>
 Coordinates:
     time     float64 201.0
 Dimensions without coordinates: segment,
 'ICV_trend_11yr': <xarray.DataArray 'trend' (segment: 163)>
 dask.array<sub, shape=(163,), dtype=float64, chunksize=(163,), chunktype=numpy.ndarray>
 Coordinates:
     time     float64 201.0
 Dimensions without coordinates: segment,
 'ICV_trend_12yr': <xarray.DataArray 'trend' (segment: 162)>
 dask.array<sub, shape=(162,), dtype=float64, chunksize=(162,), chunktype=numpy.ndarray>
 Coordinates:
     time     float64 201.0
 Dimensions without coordinates: segment,
 'ICV_trend_13yr': <xarray.DataArray 'trend' (segment: 161)>
 dask.array<sub, shape=(161,), dtype=float64, chunksize=(161,), chunktype=numpy.ndarray>
 Coordinates:
     time     float64 201.0
 Dimensions without coordinates: segment,
 'ICV_trend_14yr': <xarray.DataArray 'trend' (segment: 160)>
 da

### specify the 5 and 95 percentile values for each year step and output the 10-73yr unforced percentile timeseires for each regions

In [15]:
# define the function to calculate the percentile
def calc_percentile(da, q):
    """ Calculate the qth percentile of the data along the specified dimension.
    Args:
    da: xr.DataArray
    dim: str
    q: float
    Returns:
    xr.DataArray
    """
    # remove nans for da
    da = da.dropna(dim='segment')
    lower_percentile = np.percentile(da, q)
    upper_percentile = np.percentile(da, 100-q)
    
    return lower_percentile, upper_percentile

In [16]:
# 5%---[0]
unforced_trend_subpolar_gyre_lower_percentile = {}

# 95%---[1]
unforced_trend_subpolar_gyre_upper_percentile = {}

for key in ds_subpolar_gyre_anom.keys():
    unforced_trend_subpolar_gyre_lower_percentile[key], unforced_trend_subpolar_gyre_upper_percentile[key] = calc_percentile(ds_subpolar_gyre_anom[key], 5)
    

In [17]:
unforced_trend_subpolar_gyre_lower_percentile

{'ICV_trend_10yr': -0.6974619472598806,
 'ICV_trend_11yr': -0.5922147639151067,
 'ICV_trend_12yr': -0.5375803771279148,
 'ICV_trend_13yr': -0.5095262512620999,
 'ICV_trend_14yr': -0.4253018737377248,
 'ICV_trend_15yr': -0.406728812822973,
 'ICV_trend_16yr': -0.3969018472899658,
 'ICV_trend_17yr': -0.36665696867160286,
 'ICV_trend_18yr': -0.3738608276179545,
 'ICV_trend_19yr': -0.34353483726157485,
 'ICV_trend_20yr': -0.32516500421790084,
 'ICV_trend_21yr': -0.3068815620848402,
 'ICV_trend_22yr': -0.2811921946383576,
 'ICV_trend_23yr': -0.264556332197629,
 'ICV_trend_24yr': -0.2526010609405324,
 'ICV_trend_25yr': -0.2399243277782558,
 'ICV_trend_26yr': -0.22837502309302488,
 'ICV_trend_27yr': -0.20629825301076474,
 'ICV_trend_28yr': -0.16977942996469464,
 'ICV_trend_29yr': -0.16385206039635059,
 'ICV_trend_30yr': -0.16110980615312662,
 'ICV_trend_31yr': -0.15999280256448928,
 'ICV_trend_32yr': -0.15423825899776578,
 'ICV_trend_33yr': -0.15359745217741938,
 'ICV_trend_34yr': -0.145799229

In [18]:
# transform the dictionary to the dataarray
icv_years = list(unforced_trend_subpolar_gyre_lower_percentile.keys())
unforced_trend_subpolar_gyre_lower_percentile_da = xr.DataArray(
	list(unforced_trend_subpolar_gyre_lower_percentile.values()),
	coords={'ICV_trend_year_lower': icv_years},
	dims=['ICV_trend_year_lower'],
)
unforced_trend_subpolar_gyre_upper_percentile_da = xr.DataArray(
	list(unforced_trend_subpolar_gyre_upper_percentile.values()),
	coords={'ICV_trend_year_upper': icv_years},
	dims=['ICV_trend_year_upper'],
)
    

In [19]:
# save the percentile data
dir_out = '/work/mh0033/m301036/OBS_LPS_revision/docs/data/FIG5/data/percentile/'
# use variable names that do not conflict with coordinate names
unforced_trend_subpolar_gyre_lower_percentile_da.to_dataset(name="ICV_trend_lower").to_netcdf(dir_out+'internal_subpolar_gyre_trend_lower_percentile.nc')
unforced_trend_subpolar_gyre_upper_percentile_da.to_dataset(name="ICV_trend_upper").to_netcdf(dir_out+'internal_subpolar_gyre_trend_upper_percentile.nc')